# 🧠 End-to-End Brain MRI 4-Class Classification Framework
### Custom Baseline Attention CNN (CBAM) & Transfer Learning (ResNet50, DenseNet121, EfficientNetV2B0)

This notebook presents a complete, self-contained pipeline for 4-class brain MRI tumor diagnosis (**Glioma**, **Meningioma**, **No Tumor**, **Pituitary**).

**Key Highlights:**
- Strict **70/15/15 Stratified Split** (4,824 Train, 1,034 Val, 1,034 Test) with zero data leakage.
- **Custom Baseline Attention CNN** with Channel and Spatial CBAM attention.
- **3 Transfer Learning Architectures**: ResNet50, DenseNet121, EfficientNetV2B0.
- **2-Stage Fine-Tuning** preserving frozen Batch Normalization layers.
- **Comprehensive Metrics**: Accuracy, Precision, Sensitivity (Recall), Specificity, Macro F1, and ROC-AUC.
- **Grad-CAM Explainability** heatmaps overlay.

In [ ]:
import os
import glob
import json
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from keras import layers, models, regularizers
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Random Seed & GPU Setup
SEED = 42
tf.keras.utils.set_random_seed(SEED)
np.random.seed(SEED)

print(f"TensorFlow Version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
print(f"Available GPUs: {len(gpus)}")
for gpu in gpus:
    print(f"  • {gpu}")

## 1. Dataset Configuration & Stratified Splitting (70/15/15)
Guarantees strict isolation of training, validation, and testing sets.

In [ ]:
CLASSES = ["glioma", "meningioma", "notumor", "pituitary"]
NUM_CLASSES = len(CLASSES)
class_to_idx = {c: i for i, c in enumerate(CLASSES)}

# Locate dataset directory
DATA_DIR = "dataset " if os.path.exists("dataset ") else "."

all_paths = []
all_labels = []
for c in CLASSES:
    c_dir = os.path.join(DATA_DIR, c)
    files = glob.glob(os.path.join(c_dir, "*"))
    for f in files:
        if os.path.isfile(f) and not os.path.basename(f).startswith('.'):
            all_paths.append(f)
            all_labels.append(class_to_idx[c])

all_paths = np.array(all_paths)
all_labels = np.array(all_labels)

# Split: 70% Train, 15% Val, 15% Test
train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    all_paths, all_labels, test_size=0.30, stratify=all_labels, random_state=SEED
)
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.50, stratify=temp_labels, random_state=SEED
)

print("=" * 50)
print(f"Total Images       : {len(all_paths)}")
print(f"Training Set (70%) : {len(train_paths)}")
print(f"Validation Set(15%): {len(val_paths)}")
print(f"Test Set (15%)     : {len(test_paths)}")
print("=" * 50)

# Compute Class Weights
weights = compute_class_weight(class_weight='balanced', classes=np.unique(train_labels), y=train_labels)
class_weight_dict = {int(c): float(w) for c, w in zip(np.unique(train_labels), weights)}
print(f"Balanced Class Weights: {class_weight_dict}")

## 2. Preprocessing & Augmentation Pipelines
Data augmentation is applied **strictly to the training set**, while validation and test sets receive only static normalization.

In [ ]:
augmentation_model = tf.keras.Sequential([
    layers.RandomRotation(0.04),
    layers.RandomZoom(0.05),
    layers.RandomTranslation(height_factor=0.03, width_factor=0.03),
    layers.RandomFlip("horizontal"),
    layers.RandomContrast(0.05),
], name="augmentation_pipeline")

def preprocess_baseline(x):
    return x / 255.0

def preprocess_resnet(x):
    return tf.keras.applications.resnet50.preprocess_input(x)

def preprocess_densenet(x):
    return tf.keras.applications.densenet.preprocess_input(x)

def preprocess_efficientnet(x):
    return tf.keras.applications.efficientnet_v2.preprocess_input(x)

def parse_image(path, label, img_size=(224, 224)):
    img_bytes = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img_bytes, channels=3)
    img = tf.image.resize(img, img_size)
    img = tf.cast(img, tf.float32)
    return img, tf.one_hot(label, NUM_CLASSES)

def build_tf_dataset(paths, labels, batch_size=32, is_training=False, pfn=None, img_size=(224, 224)):
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if is_training:
        dataset = dataset.shuffle(buffer_size=len(paths), reshuffle_each_iteration=True)
    dataset = dataset.map(lambda p, l: parse_image(p, l, img_size=img_size), num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.cache()
    if is_training:
        dataset = dataset.map(lambda x, y: (augmentation_model(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    if pfn is not None:
        dataset = dataset.map(lambda x, y: (pfn(x), y), num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return dataset

## 3. Model Architecture Definitions
- **Model 1**: Baseline Custom CNN with CBAM Attention
- **Model 2**: ResNet50
- **Model 3**: DenseNet121
- **Model 4**: EfficientNetV2B0

In [ ]:
def apply_cbam_attention(input_tensor, ratio=8):
    # Channel Attention
    channel = input_tensor.shape[-1]
    avg_pool = layers.GlobalAveragePooling2D(keepdims=True)(input_tensor)
    max_pool = layers.GlobalMaxPooling2D(keepdims=True)(input_tensor)
    mlp = models.Sequential([
        layers.Dense(channel // ratio, activation='relu', use_bias=False),
        layers.Dense(channel, use_bias=False)
    ])
    channel_attention = layers.Activation('sigmoid')(mlp(avg_pool) + mlp(max_pool))
    x = layers.Multiply()([input_tensor, channel_attention])
    
    # Spatial Attention
    avg_spatial = tf.reduce_mean(x, axis=-1, keepdims=True)
    max_spatial = tf.reduce_max(x, axis=-1, keepdims=True)
    concat = layers.Concatenate(axis=-1)([avg_spatial, max_spatial])
    spatial_attention = layers.Conv2D(1, kernel_size=7, padding='same', activation='sigmoid')(concat)
    return layers.Multiply()([x, spatial_attention])

def build_baseline_attention_cnn(input_shape=(256, 256, 3), num_classes=4, weight_decay=1e-4):
    inputs = layers.Input(shape=input_shape)
    x = inputs
    filters = [32, 64, 128, 256, 512]
    for i, f in enumerate(filters):
        x = layers.Conv2D(f, (3, 3), padding='same', kernel_regularizer=regularizers.l2(weight_decay), name=f"conv{i+1}_1")(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.Conv2D(f, (3, 3), padding='same', kernel_regularizer=regularizers.l2(weight_decay), name=f"conv{i+1}_2")(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        if i in [2, 4]:
            x = apply_cbam_attention(x)
        x = layers.MaxPooling2D((2, 2))(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(weight_decay))(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return models.Model(inputs=inputs, outputs=outputs, name="Baseline_Attention_CNN")

def build_resnet50(input_shape=(224, 224, 3), num_classes=4):
    inputs = layers.Input(shape=input_shape)
    backbone = tf.keras.applications.ResNet50(weights='imagenet', include_top=False, input_tensor=inputs)
    for layer in backbone.layers:
        layer.trainable = False
    x = backbone.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    model = models.Model(inputs=inputs, outputs=outputs, name="ResNet50")
    model.base_model = backbone
    return model

def build_densenet121(input_shape=(224, 224, 3), num_classes=4):
    inputs = layers.Input(shape=input_shape)
    backbone = tf.keras.applications.DenseNet121(weights='imagenet', include_top=False, input_tensor=inputs)
    for layer in backbone.layers:
        layer.trainable = False
    x = backbone.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    model = models.Model(inputs=inputs, outputs=outputs, name="DenseNet121")
    model.base_model = backbone
    return model

def build_efficientnetv2b0(input_shape=(224, 224, 3), num_classes=4):
    inputs = layers.Input(shape=input_shape)
    backbone = tf.keras.applications.EfficientNetV2B0(weights='imagenet', include_top=False, input_tensor=inputs)
    for layer in backbone.layers:
        layer.trainable = False
    x = backbone.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation='swish', kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation='swish', kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    model = models.Model(inputs=inputs, outputs=outputs, name="EfficientNetV2B0")
    model.base_model = backbone
    return model

## 4. Evaluation & Metric Calculation Functions
Computes Accuracy, Precision, Sensitivity (Recall), Specificity, F1 Score, and ROC-AUC.

In [ ]:
def calc_specificity(cm):
    specs = []
    for i in range(4):
        tp = cm[i, i]
        fn = np.sum(cm[i, :]) - tp
        fp = np.sum(cm[:, i]) - tp
        tn = np.sum(cm) - (tp + fn + fp)
        spec = tn / (tn + fp)
        specs.append(spec)
    return np.mean(specs)

def evaluate_model_comprehensive(model, test_ds, test_labels, model_name="Model"):
    preds = model.predict(test_ds, verbose=0)
    pred_labels = np.argmax(preds, axis=1)
    
    acc = accuracy_score(test_labels, pred_labels)
    prec = precision_score(test_labels, pred_labels, average='macro')
    rec = recall_score(test_labels, pred_labels, average='macro')
    f1_m = f1_score(test_labels, pred_labels, average='macro')
    auc_val = roc_auc_score(test_labels, preds, multi_class='ovr', average='macro')
    
    cm = confusion_matrix(test_labels, pred_labels)
    spec = calc_specificity(cm)
    
    print(f"=== {model_name} Final Test Evaluation ===")
    print(f"Accuracy    : {acc*100:.2f}%")
    print(f"Precision   : {prec*100:.2f}%")
    print(f"Sensitivity : {rec*100:.2f}%")
    print(f"Specificity : {spec*100:.2f}%")
    print(f"Macro F1    : {f1_m*100:.2f}%")
    print(f"ROC-AUC     : {auc_val:.4f}\n")
    
    return {
        "Model": model_name,
        "Accuracy (%)": f"{acc*100:.2f}%",
        "Precision (%)": f"{prec*100:.2f}%",
        "Sensitivity (%)": f"{rec*100:.2f}%",
        "Specificity (%)": f"{spec*100:.2f}%",
        "F1 Score (%)": f"{f1_m*100:.2f}%",
        "ROC-AUC": f"{auc_val:.4f}"
    }

## 5. Grad-CAM Explainability Heatmaps Generator
Visualizes local spatial feature activation overlay for all target tumor classes.

In [ ]:
def get_last_conv_layer(model):
    for layer in reversed(model.layers):
        if isinstance(layer, (tf.keras.layers.Conv2D, tf.keras.layers.DepthwiseConv2D)):
            return layer.name
    return None

def generate_gradcam(model, img_batch, target_layer_name, pred_index=None):
    target_layer = model.get_layer(target_layer_name)
    grad_model = tf.keras.models.Model(inputs=[model.inputs], outputs=[target_layer.output, model.output])
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_batch)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]
    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = conv_outputs[0] @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-10)
    return heatmap.numpy()